In [1]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd


In [3]:
data_processed = '/content/drive/MyDrive/UMD/DATA606/p2/preprocessed/ratings_books_users.csv'

df = pd.read_csv(data_processed)
df = df[df['Book-Rating'] > 0]  # keep only explicit ratings
df = df[['User-ID', 'ISBN', 'Book-Rating', 'Country', 'Publisher', 'Book-Author', 'Year-Of-Publication']].dropna()

In [4]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.25, random_state=42)

# Ensure cold-start items/users are not in test
train_users = set(train_df['User-ID'])
train_items = set(train_df['ISBN'])
test_df = test_df[test_df['User-ID'].isin(train_users) & test_df['ISBN'].isin(train_items)]

In [5]:
from scipy.sparse import csr_matrix

train_matrix = train_df.pivot(index='ISBN', columns='User-ID', values='Book-Rating').fillna(0)
train_sparse = csr_matrix(train_matrix.values)
isbn_list = train_matrix.index.tolist()
user_list = train_matrix.columns.tolist()

<ipython-input-5-221bfe43adda>:3: PerformanceWarning: The following operation may generate 6895374892 cells in the resulting pandas object.
  train_matrix = train_df.pivot(index='ISBN', columns='User-ID', values='Book-Rating').fillna(0)


In [6]:
from sklearn.neighbors import NearestNeighbors

knn_model = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=50, n_jobs=-1)
knn_model.fit(train_sparse)

NearestNeighbors(algorithm='brute', metric='cosine', n_jobs=-1, n_neighbors=50)

In [7]:
import numpy as np

def predict_rating(user_id, item_isbn, k=10):
    if item_isbn not in isbn_list or user_id not in user_list:
        return np.nan  # cold start

    item_idx = isbn_list.index(item_isbn)
    user_idx = user_list.index(user_id)

    # Find K nearest items to the target item
    distances, indices = knn_model.kneighbors(train_sparse[item_idx], n_neighbors=k+1)
    sim_scores = 1 - distances.flatten()[1:]
    neighbor_indices = indices.flatten()[1:]

    # Extract ratings the user gave to neighbor items
    user_ratings = train_matrix.iloc[neighbor_indices, user_idx].values

    mask = user_ratings > 0
    if not np.any(mask):
        return np.nan

    sim_scores = sim_scores[mask]
    user_ratings = user_ratings[mask]

    pred = np.dot(sim_scores, user_ratings) / np.sum(sim_scores)
    return pred

In [8]:

from tqdm import tqdm
from sklearn.metrics import mean_absolute_error

y_true, y_pred = [], []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    uid, isbn, true_rating = row['User-ID'], row['ISBN'], row['Book-Rating']
    pred = predict_rating(uid, isbn, k=10)

    if not np.isnan(pred):
        y_true.append(true_rating)
        y_pred.append(pred)

mad_score = mean_absolute_error(y_true, y_pred)


  2%|▏         | 931/58572 [02:24<4:44:23,  3.38it/s]<ipython-input-7-13e8860d69b5>:25: RuntimeWarning: invalid value encountered in scalar divide
  pred = np.dot(sim_scores, user_ratings) / np.sum(sim_scores)
100%|██████████| 58572/58572 [2:32:20<00:00,  6.41it/s]

Mean Absolute Difference (MAD): 0.7446


In [9]:

import pandas as pd
import numpy as np

# Save dataframes to csv files
train_df.to_csv('/content/drive/MyDrive/UMD/DATA606/p2/preprocessed/x_train.csv', index=False)
test_df.to_csv('/content/drive/MyDrive/UMD/DATA606/p2/preprocessed/x_test.csv', index=False)

# Convert lists to dataframes before saving
y_train_df = pd.DataFrame({'y_train': y_true})
y_train_df.to_csv('/content/drive/MyDrive/UMD/DATA606/p2/preprocessed/y_train.csv', index=False)

y_test_df = pd.DataFrame({'y_test': y_true})
y_test_df.to_csv('/content/drive/MyDrive/UMD/DATA606/p2/preprocessed/y_test.csv', index=False)

y_pred_df = pd.DataFrame({'y_pred': y_pred})
y_pred_df.to_csv('/content/drive/MyDrive/UMD/DATA606/p2/preprocessed/y_pred.csv', index=False)


In [10]:
print(f"Mean Absolute Error (MAE): {mad_score:.4f}")

Mean Absolute Error (MAE): 0.7446


In [15]:
# Assuming your data is in a CSV file named 'ratings.csv'
data = pd.read_csv('/content/drive/MyDrive/UMD/DATA606/p2/preprocessed/ratings_books_users.csv')

# Keep only explicit ratings (e.g., ratings greater than 0)
data = data[data['Book-Rating'] > 0]

# Select relevant columns (user, item, rating)
data = data[['User-ID', 'ISBN', 'Book-Rating']]

In [16]:
data.head()

,User-ID,ISBN,Book-Rating
1,276726,0155061224,5
3,276729,052165615X,3
4,276729,0521795028,6
6,276744,038550120X,7
13,276747,0060517794,9


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import mean_absolute_error
from tqdm import tqdm

# 1. Data Loading and Preprocessing
data = pd.read_csv('/content/drive/MyDrive/UMD/DATA606/p2/preprocessed/ratings_books_users.csv')
data = data[data['Book-Rating'] > 0]
data = data[['User-ID', 'ISBN', 'Book-Rating']]

# 2. Data Splitting
train_data, test_data = train_test_split(data, test_size=0.25, random_state=42)
train_users = set(train_data['User-ID'])
train_items = set(train_data['ISBN'])
test_data = test_data[test_data['User-ID'].isin(train_users) & test_data['ISBN'].isin(train_items)]

# 3. Building the Item-Item Similarity Matrix
train_matrix = train_data.pivot(index='ISBN', columns='User-ID', values='Book-Rating').fillna(0)
train_sparse = csr_matrix(train_matrix.values)
knn_model = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=10, n_jobs=-1)
knn_model.fit(train_sparse)

# 4. Prediction Function
def predict_rating(user_id, item_isbn, k=10):
    try:
        item_idx = train_matrix.index.get_loc(item_isbn)
    except KeyError:
        return np.nan  # Cold-start item

    distances, indices = knn_model.kneighbors(train_sparse[item_idx], n_neighbors=k+1)
    sim_scores = 1 - distances.flatten()[1:]
    neighbor_indices = indices.flatten()[1:]

    user_ratings = train_matrix.iloc[neighbor_indices, train_matrix.columns.get_loc(user_id)].values

    mask = user_ratings > 0
    if not np.any(mask):
        return np.nan

    sim_scores = sim_scores[mask]
    user_ratings = user_ratings[mask]

    pred = np.dot(sim_scores, user_ratings) / np.sum(sim_scores)
    return pred

# 5. Evaluation
y_true, y_pred = [], []

for _, row in tqdm(test_data.iterrows(), total=len(test_data)):
    uid, isbn, true_rating = row['User-ID'], row['ISBN'], row['Book-Rating']
    pred = predict_rating(uid, isbn, k=10)

    if not np.isnan(pred):
        y_true.append(true_rating)
        y_pred.append(pred)

mae = mean_absolute_error(y_true, y_pred)
print(f"Mean Absolute Error (MAE): {mae:.4f}")

<ipython-input-18-b3bebfa6e12c>:21: PerformanceWarning: The following operation may generate 6895374892 cells in the resulting pandas object.
  train_matrix = train_data.pivot(index='ISBN', columns='User-ID', values='Book-Rating').fillna(0)
  2%|▏         | 931/58572 [02:12<2:14:51,  7.12it/s]<ipython-input-18-b3bebfa6e12c>:46: RuntimeWarning: invalid value encountered in scalar divide
  pred = np.dot(sim_scores, user_ratings) / np.sum(sim_scores)
100%|██████████| 58572/58572 [2:23:23<00:00,  6.81it/s]

Mean Absolute Error (MAE): 0.7446
